<a href="https://colab.research.google.com/github/nataliamarinn/labo3-2026r/blob/main/src/AutoGluon/z329_ErrorIrreducible.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Error Irreducible — ¿Qué parte del error no se puede corregir?

## El problema

En el backtesting vimos que para algunos productos **todos los modelos se equivocan en la misma dirección** y por un margen enorme:

```
product 20008: real=195  | naive=469  har=444  arima=419  ag=390
product 20014: real=272  | naive=479  har=434  arima=455  ag=390
```

Eso **no es ruido aleatorio**. Cuando todos los modelos sobre-predicen por el mismo factor (~2x), hay un quiebre estructural: el producto cayó a la mitad de su nivel histórico y ningún modelo lo anticipó.

## Dos tipos de error

| Tipo | Causa | ¿Corregible? |
|---|---|---|
| **Ruido aleatorio** | La serie siempre fue volátil — varianza inherente | No, es la varianza irreducible de la serie |
| **Quiebre estructural** | El proceso generador de datos cambió — caída abrupta, descontinuación | No desde la historia sola. Necesita info externa |
| **Error de modelo** | El modelo no captura la estructura que sí existe en la serie | Sí — mejorar el modelo |

## ¿Cómo los separamos?

Para cada producto calculamos:

1. **R² in-sample del HAR**: si R² es alto, la serie tiene estructura predecible. Si R² es bajo, la serie es ruidosa de base.
2. **Factor sorpresa** = (tn_201912 - media_historia) / std_historia. Si > 2σ → quiebre estructural.
3. **Dirección del error**: si todos los modelos sobre-predicen → caída abrupta. Si predicen mezclado → ruido.

### Cuadrantes de diagnóstico

```
                    Error grande    Error chico
                 ┌──────────────┬─────────────┐
Serie            │  QUIEBRE     │  BIEN       │
estructurada     │  ESTRUCTURAL │  MODELADO   │
(R² alto)        │  irreducible │             │
                 ├──────────────┼─────────────┤
Serie            │  RUIDO       │  RUIDO      │
no estructurada  │  ESPERADO    │  ESPERADO   │
(R² bajo)        │  normal      │  (suerte)   │
                 └──────────────┴─────────────┘
```

El cuadrante superior-izquierdo (estructurada + error grande) = **error irreducible por quiebre**.

## 0.1 Init ambiente Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/.drive')

In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/labo3"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/labo3"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json

mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets

descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/austral2026-5da5/labo3/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

descargar  "sell-in.txt.gz"
descargar  "tb_productos.txt"
descargar  "tb_stocks.txt"
descargar  "product_id_apredecir201912.txt"

# 1  Setup

In [ ]:
!pip install uv
!uv pip install -q kaggle

In [ ]:
import os
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
PARAM = {
  'experimento':   'ErrorIrreducible-01',
  'periodo_corte': 201910,
  'periodo_target': 201912,
  # umbral de sorpresa: cuántas sigmas de diferencia respecto a la media histórica
  'sigma_quiebre': 2.0,
  # umbral de R² para considerar serie "estructurada"
  'r2_estructura': 0.3,
  # ruta del CSV de errores del backtesting (z326)
  'csv_errores': '/content/buckets/b1/exp/Backtesting-01/backtesting_errores_201912.csv'
}

In [ ]:
ruta = "/content/buckets/b1/exp/" + PARAM['experimento']
os.makedirs(ruta, exist_ok=True)
os.chdir(ruta)

# 2  Datos

In [ ]:
dataset = pl.read_csv('/content/.drive/My Drive/labo3/datasets/sell-in.txt.gz', separator="\t")

tb_ventas = dataset.group_by("product_id", "periodo").agg(
    pl.col("tn").sum().alias("tn")
).sort(["product_id", "periodo"])

tb_apredecir = pl.read_csv('/content/.drive/My Drive/labo3/datasets/product_id_apredecir201912.txt', separator="\t")
tb_ventas    = tb_ventas.join(tb_apredecir, on="product_id", how="inner").sort(["product_id", "periodo"])

tb_train = tb_ventas.filter(pl.col("periodo") <= PARAM['periodo_corte'])
tb_real  = tb_ventas.filter(pl.col("periodo") == PARAM['periodo_target']).select(["product_id", "tn"]).rename({"tn": "tn_real"})

tb_errores = pl.read_csv(PARAM['csv_errores'])

productos = tb_apredecir["product_id"].to_list()
print(f"{len(productos)} productos  |  errores cargados: {tb_errores.height} filas")

# 3  Diagnóstico por producto

Para cada producto calculamos:

| Métrica | Fórmula | Qué mide |
|---|---|---|
| `r2_har` | R² del HAR en-muestra | ¿Cuánta estructura predecible tenía la serie? |
| `cv` | std / mean | Volatilidad relativa histórica |
| `sorpresa` | (tn_real - media_train) / std_train | ¿Cuántas sigmas se alejó 201912 de la historia? |
| `direccion` | signo promedio de (pred - real) | +1 = todos sobre-predicen, -1 = todos sub-predicen, ~0 = mixto |
| `consenso` | std de los errores entre modelos / media errores | Si es bajo, todos se equivocan igual → quiebre |
| `nivel_caida` | tn_real / media_train | 1 = sin cambio, <0.5 = cayó a la mitad |

In [ ]:
def build_har_features(serie):
    T = len(serie)
    rows_X, rows_y = [], []
    for t in range(12, T):
        rows_X.append([serie[t-1], serie[t-3:t].mean(), serie[t-6:t].mean(), serie[t-12:t].mean()])
        rows_y.append(serie[t])
    return np.array(rows_X), np.array(rows_y)


diagnostico = []

for pid in productos:
    serie = (
        tb_train.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )
    real = float(tb_real.filter(pl.col("product_id") == pid)["tn_real"][0])
    row_err = tb_errores.filter(pl.col("product_id") == pid)

    media_train = serie.mean()
    std_train   = serie.std() + 1e-9
    cv          = std_train / (media_train + 1e-9)

    # R² HAR in-sample
    r2 = np.nan
    if len(serie) >= 14:
        try:
            X, y = build_har_features(serie)
            fitted = LinearRegression().fit(X, y).predict(X)
            r2 = float(r2_score(y, fitted))
        except Exception:
            pass

    # sorpresa: cuántas sigmas se alejó el real de la media histórica
    sorpresa = (real - media_train) / std_train

    # nivel de caída relativo
    nivel_caida = real / (media_train + 1e-9)

    # dirección y consenso del error entre modelos
    preds = np.array([
        float(row_err["pred_naive"][0]),
        float(row_err["pred_har"][0]),
        float(row_err["pred_arima"][0]),
        float(row_err["pred_ag"][0]),
    ])
    errores_sig = preds - real  # positivo = sobre-predicción
    direccion   = float(np.mean(errores_sig))          # > 0: todos sobre-predicen
    consenso    = float(np.std(errores_sig) / (np.mean(np.abs(errores_sig)) + 1e-9))  # bajo = acuerdo total
    err_promedio = float(row_err["err_promedio"][0])

    diagnostico.append({
        'product_id':   pid,
        'tn_real':      real,
        'media_train':  float(media_train),
        'std_train':    float(std_train),
        'cv':           float(cv),
        'r2_har':       r2,
        'sorpresa':     float(sorpresa),
        'nivel_caida':  float(nivel_caida),
        'direccion':    direccion,
        'consenso':     float(consenso),
        'err_promedio': err_promedio,
    })

tb_diag = pl.DataFrame(diagnostico)
print(f"Diagnóstico completo: {tb_diag.height} productos")
display(tb_diag.sort('err_promedio', descending=True).head(10))

# 4  Clasificación de productos

Asignamos cada producto a una categoría según los ejes R² (estructura) y sorpresa (quiebre).

In [ ]:
tb_diag = tb_diag.with_columns([
    # tiene estructura predecible?
    (pl.col('r2_har') >= PARAM['r2_estructura']).alias('es_estructurado'),
    # fue una sorpresa estadística?
    (pl.col('sorpresa').abs() >= PARAM['sigma_quiebre']).alias('es_quiebre'),
    # todos los modelos sobre-predicen? (dirección positiva y consenso bajo = acuerdo)
    ((pl.col('direccion') > 0) & (pl.col('consenso') < 0.3)).alias('sobreprediccion_consenso'),
])

# etiqueta de cuadrante
def etiquetar(es_struct, es_quiebre):
    if es_struct and es_quiebre:
        return 'QUIEBRE_ESTRUCTURAL'
    elif es_struct and not es_quiebre:
        return 'BIEN_MODELADO'
    elif not es_struct and es_quiebre:
        return 'RUIDO_QUIEBRE'
    else:
        return 'RUIDO_NORMAL'

etiquetas = [
    etiquetar(bool(r['es_estructurado']), bool(r['es_quiebre']))
    for r in tb_diag.iter_rows(named=True)
]
tb_diag = tb_diag.with_columns(pl.Series('categoria', etiquetas))

print("Distribución de categorías:")
conteo = tb_diag.group_by('categoria').agg([
    pl.len().alias('n_productos'),
    pl.col('err_promedio').mean().alias('err_medio'),
    pl.col('err_promedio').sum().alias('err_total'),
]).sort('err_total', descending=True)
display(conteo)

# % del error total que viene de cada categoría
err_total_global = tb_diag['err_promedio'].sum()
print(f"\n% del error total por categoría:")
for row in conteo.iter_rows(named=True):
    pct = row['err_total'] / err_total_global * 100
    print(f"  {row['categoria']:25s}: {pct:5.1f}%  ({row['n_productos']} productos)")

# 5  Visualización: cuadrante R² vs Sorpresa

Cada punto es un producto. El tamaño es proporcional al error promedio entre modelos.

In [ ]:
colores_cat = {
    'QUIEBRE_ESTRUCTURAL': 'tomato',
    'BIEN_MODELADO':       'steelblue',
    'RUIDO_QUIEBRE':       'darkorange',
    'RUIDO_NORMAL':        'lightgray',
}

df_plot = tb_diag.to_pandas()
df_plot = df_plot.dropna(subset=['r2_har'])

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- izquierda: R² vs sorpresa ---
for cat, color in colores_cat.items():
    sub = df_plot[df_plot['categoria'] == cat]
    size = np.clip(sub['err_promedio'] * 2, 10, 300)
    axes[0].scatter(sub['sorpresa'], sub['r2_har'],
                    c=color, s=size, alpha=0.6, label=cat, edgecolors='white', linewidth=0.3)

axes[0].axvline(-PARAM['sigma_quiebre'], color='red', linestyle='--', linewidth=0.8, label=f'±{PARAM["sigma_quiebre"]}σ')
axes[0].axvline( PARAM['sigma_quiebre'], color='red', linestyle='--', linewidth=0.8)
axes[0].axhline(PARAM['r2_estructura'], color='navy', linestyle='--', linewidth=0.8, label=f'R²={PARAM["r2_estructura"]}')
axes[0].set_xlabel('Sorpresa (sigmas respecto a media histórica)', fontsize=9)
axes[0].set_ylabel('R² HAR in-sample (estructura predecible)', fontsize=9)
axes[0].set_title('Diagnóstico por producto\n(tamaño = error promedio entre modelos)', fontsize=9)
axes[0].legend(fontsize=7)

# anotaciones de cuadrantes
axes[0].text(-6, 0.85, 'QUIEBRE\nESTRUCTURAL\nirreducible', ha='center', fontsize=7,
             color='tomato', fontweight='bold')
axes[0].text(5, 0.85, 'QUIEBRE\nEST. (subida)', ha='center', fontsize=7,
             color='tomato', alpha=0.6)
axes[0].text(0, 0.85, 'BIEN\nMODELADO', ha='center', fontsize=7, color='steelblue')
axes[0].text(0, 0.05, 'RUIDO NORMAL', ha='center', fontsize=7, color='gray')

# --- derecha: nivel_caida vs err_promedio coloreado por categoría ---
for cat, color in colores_cat.items():
    sub = df_plot[df_plot['categoria'] == cat]
    axes[1].scatter(sub['nivel_caida'], sub['err_promedio'],
                    c=color, s=25, alpha=0.6, label=cat, edgecolors='white', linewidth=0.3)

axes[1].axvline(1.0, color='black', linestyle='--', linewidth=0.8, label='nivel sin cambio')
axes[1].axvline(0.5, color='red',   linestyle=':',  linewidth=0.8, label='cayó al 50%')
axes[1].set_xlabel('Nivel real / media histórica  (1=sin cambio, 0.5=cayó a la mitad)', fontsize=9)
axes[1].set_ylabel('Error promedio entre modelos', fontsize=9)
axes[1].set_title('Caída de nivel vs error\n(rojo=quiebre estructural)', fontsize=9)
axes[1].set_xlim(left=0)
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

# 6  ¿Cuánto del error total es irreducible?

Descomponemos el RMSE global en la contribución de cada categoría.

In [ ]:
modelos_cols = ['err_naive', 'err_har', 'err_arima', 'err_ag']
tb_full = tb_diag.join(tb_errores.select(['product_id'] + modelos_cols), on='product_id', how='left')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cats_orden = ['QUIEBRE_ESTRUCTURAL', 'RUIDO_QUIEBRE', 'RUIDO_NORMAL', 'BIEN_MODELADO']
colores_bar = ['tomato', 'darkorange', 'lightgray', 'steelblue']

# RMSE por categoría × modelo
rmse_cat = {}
for cat in cats_orden:
    sub = tb_full.filter(pl.col('categoria') == cat)
    rmse_cat[cat] = {}
    for m in ['naive', 'har', 'arima', 'ag']:
        errs = sub[f'err_{m}'].to_numpy()
        rmse_cat[cat][m] = float(np.sqrt((errs**2).mean())) if len(errs) > 0 else 0.0

x = np.arange(4)
width = 0.18
modelos_label = ['Naive', 'HAR', 'ARIMA', 'AutoGluon']

for i, (cat, color) in enumerate(zip(cats_orden, colores_bar)):
    vals = [rmse_cat[cat][m] for m in ['naive', 'har', 'arima', 'ag']]
    axes[0].bar(x + i * width, vals, width, label=cat, color=color, alpha=0.85)

axes[0].set_xticks(x + width * 1.5)
axes[0].set_xticklabels(modelos_label)
axes[0].set_ylabel('RMSE')
axes[0].set_title('RMSE por modelo y categoría de producto')
axes[0].legend(fontsize=7)

# % de error total aportado por cada categoría (para AutoGluon)
err_ag_total = tb_full['err_ag'].to_numpy()
sse_total = (err_ag_total**2).sum()

pcts = []
for cat in cats_orden:
    sub = tb_full.filter(pl.col('categoria') == cat)
    sse_cat = (sub['err_ag'].to_numpy()**2).sum()
    pcts.append(sse_cat / sse_total * 100)

axes[1].barh(cats_orden, pcts, color=colores_bar, alpha=0.85)
for i, v in enumerate(pcts):
    axes[1].text(v + 0.3, i, f'{v:.1f}%', va='center', fontsize=9)
axes[1].set_xlabel('% del SSE total (AutoGluon)')
axes[1].set_title('¿Cuánto del error total viene de cada categoría?')
axes[1].axvline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.show()

print("\nResumen:")
q_pct = pcts[cats_orden.index('QUIEBRE_ESTRUCTURAL')]
print(f"  Error irreducible por quiebre estructural: {q_pct:.1f}% del SSE total")
print(f"  Error en series bien modeladas:           {pcts[cats_orden.index('BIEN_MODELADO')]:.1f}% del SSE total")

# 7  Series con quiebre estructural — ¿qué pasó?

Para los productos en `QUIEBRE_ESTRUCTURAL` mostramos la serie completa con una señal adicional:
la **tendencia de los últimos 6 meses antes del corte**.

Si el quiebre era visible desde antes (tendencia negativa pronunciada), el modelo lo podría haber capturado con más peso al lag reciente. Si es abrupto (caída en el último mes), es genuinamente imprevisible.

In [ ]:
pids_quiebre = (
    tb_diag
    .filter(pl.col('categoria') == 'QUIEBRE_ESTRUCTURAL')
    .sort('err_promedio', descending=True)
    .head(9)
    ['product_id'].to_list()
)

n = len(pids_quiebre)
ncols = 3
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = axes.flatten() if n > 1 else [axes]

for i, pid in enumerate(pids_quiebre):
    serie_full = tb_ventas.filter(pl.col('product_id') == pid).sort('periodo')
    periodos   = serie_full['periodo'].to_list()
    tn         = serie_full['tn'].to_numpy().astype(float)

    # split train / target
    idx_corte  = next((j for j, p in enumerate(periodos) if p > PARAM['periodo_corte']), len(periodos))
    idx_target = next((j for j, p in enumerate(periodos) if p == PARAM['periodo_target']), None)

    tn_train  = tn[:idx_corte]
    x_train   = np.arange(len(tn_train))

    # tendencia lineal sobre los últimos 6 meses de train
    ult6  = tn_train[-6:]
    x6    = np.arange(len(tn_train) - 6, len(tn_train))
    slope = np.polyfit(x6, ult6, 1)[0]
    tendencia_str = f'↓{abs(slope):.1f}/mes' if slope < 0 else f'↑{slope:.1f}/mes'

    # predicciones de los 4 modelos
    row_err  = tb_errores.filter(pl.col('product_id') == pid)
    row_diag = tb_diag.filter(pl.col('product_id') == pid)
    sorpresa = float(row_diag['sorpresa'][0])

    ax = axes[i]
    ax.plot(x_train, tn_train, 'o-', color='steelblue', markersize=3, linewidth=1.5, label='historia train')

    # linea de tendencia reciente (últimos 6 meses)
    x6_ext  = np.array([len(tn_train) - 6, len(tn_train) + 1])
    y6_ext  = np.polyval(np.polyfit(x6, ult6, 1), x6_ext)
    ax.plot(x6_ext, y6_ext, 'r--', linewidth=1.2, alpha=0.7, label=f'tendencia 6m ({tendencia_str})')

    if idx_target is not None:
        ax.scatter([idx_target], [tn[idx_target]], color='black', s=80, zorder=5, label=f'real={tn[idx_target]:.1f}')

    # predicciones como puntos en x = idx_target
    if idx_target is not None:
        for m_label, m_col, m_color in [('naive','pred_naive','gray'), ('HAR','pred_har','green'),
                                         ('ARIMA','pred_arima','orange'), ('AG','pred_ag','red')]:
            pred_val = float(row_err[m_col][0])
            ax.scatter([idx_target], [pred_val], color=m_color, s=40, alpha=0.7, marker='D')

    ax.set_title(f'pid {pid}  |  sorpresa={sorpresa:.1f}σ', fontsize=8)
    ax.legend(fontsize=6)
    ax.set_xlabel('períodos', fontsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle(
    'Quiebres estructurales: historia + tendencia reciente + predicciones vs real\n'
    '(diamantes = predicciones, círculo negro = real 201912)',
    fontsize=10
)
plt.tight_layout()
plt.show()

# 8  ¿Era visible el quiebre antes?

Comparamos la tendencia de los últimos 6 meses de train con el error.

Si los productos con quiebre ya tenían **pendiente negativa pronunciada**, el error era parcialmente evitable (un modelo que da más peso al lag reciente habría captado la caída).

Si la pendiente era plana o positiva y el quiebre fue abrupto en los últimos 2 meses → genuinamente irreducible.

In [ ]:
pendientes = []
for pid in productos:
    serie = (
        tb_train.filter(pl.col("product_id") == pid)
        .sort("periodo")["tn"].to_numpy().astype(float)
    )
    if len(serie) >= 6:
        ult6  = serie[-6:]
        slope = float(np.polyfit(np.arange(6), ult6, 1)[0])
        # pendiente normalizada por la media
        slope_norm = slope / (serie.mean() + 1e-9)
    else:
        slope_norm = 0.0
    pendientes.append({'product_id': pid, 'pendiente_norm_6m': slope_norm})

tb_pend = pl.DataFrame(pendientes)
tb_diag = tb_diag.join(tb_pend, on='product_id', how='left')

# solo quiebres estructurales
sub_q = tb_diag.filter(pl.col('categoria') == 'QUIEBRE_ESTRUCTURAL').to_pandas()
sub_resto = tb_diag.filter(pl.col('categoria') != 'QUIEBRE_ESTRUCTURAL').to_pandas()

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(sub_resto['pendiente_norm_6m'], sub_resto['sorpresa'],
           c='lightgray', s=15, alpha=0.5, label='otros')
sc = ax.scatter(sub_q['pendiente_norm_6m'], sub_q['sorpresa'],
                c=sub_q['err_promedio'], cmap='Reds', s=60, alpha=0.8,
                label='quiebre estructural', edgecolors='black', linewidth=0.3)
plt.colorbar(sc, label='error promedio')

ax.axhline(-PARAM['sigma_quiebre'], color='red', linestyle='--', linewidth=0.8)
ax.axvline(0, color='black', linestyle='--', linewidth=0.5)
ax.set_xlabel('Pendiente normalizada últimos 6 meses de train  (negativa = caída)', fontsize=9)
ax.set_ylabel('Sorpresa en 201912 (sigmas)', fontsize=9)
ax.set_title(
    'Quiebres estructurales: ¿la caída ya era visible en los últimos 6 meses?\n'
    'Izquierda = caída ya en curso  |  Centro/derecha = quiebre abrupto',
    fontsize=9
)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

# separación: abrupto vs gradual
n_gradual = (sub_q['pendiente_norm_6m'] < -0.05).sum()
n_abrupto = (sub_q['pendiente_norm_6m'] >= -0.05).sum()
print(f"Quiebres con caída ya visible (pendiente < -5%/mes): {n_gradual}")
print(f"Quiebres abruptos (sin señal previa):                {n_abrupto}")

# 9  Resumen final

Tabla de diagnóstico exportable con todos los indicadores por producto.

In [ ]:
tb_resumen = tb_diag.select([
    'product_id', 'categoria', 'tn_real', 'media_train',
    'r2_har', 'cv', 'sorpresa', 'nivel_caida',
    'pendiente_norm_6m', 'direccion', 'consenso', 'err_promedio'
]).sort('err_promedio', descending=True)

tb_resumen.write_csv('diagnostico_error_irreducible.csv')
print("Guardado: diagnostico_error_irreducible.csv")

print("\n--- CONCLUSIÓN ---")
total_prods = tb_diag.height
for cat in cats_orden:
    sub = tb_diag.filter(pl.col('categoria') == cat)
    n   = sub.height
    err_med = sub['err_promedio'].mean()
    print(f"  {cat:25s}: {n:4d} productos ({n/total_prods*100:.1f}%)  |  error medio = {err_med:.2f}")

print()
q_count = tb_diag.filter(pl.col('categoria') == 'QUIEBRE_ESTRUCTURAL').height
print(f"→ {q_count} productos ({q_count/total_prods*100:.1f}%) tienen error IRREDUCIBLE por quiebre estructural.")
print(f"→ Mejorar el modelo solo puede ayudar en BIEN_MODELADO y parcialmente en RUIDO_QUIEBRE.")